In [41]:
from pathlib import Path
from nautilus_trader.persistence.catalog import ParquetDataCatalog
import pandas as pd

parent_dir = Path.cwd().parent

## Initializing Catalog

In [11]:
CATALOG_PATH = parent_dir / "catalog"
catalog = ParquetDataCatalog(CATALOG_PATH)

## Loading Data

In this example:
1. Downloaded csv file doesn't have header row. Specify column names with `name`.
2. Using BarDataWrangler to process dataframe

Note: Some columns are not needed by Nautilus Trader

Binance Kiline Data Format
```json
[
  [
    1499040000000,      // Open time
    "0.01634790",       // Open
    "0.80000000",       // High
    "0.01575800",       // Low
    "0.01577100",       // Close
    "148976.11427815",  // Volume
    1499644799999,      // Close time
    "2434.19055334",    // Quote asset volume
    308,                // Number of trades
    "1756.87402397",    // Taker buy base asset volume
    "28.46694368",      // Taker buy quote asset volume
    "17928899.62484339" // Ignore.
  ]
]
```
Nautilus Trader's Bar Data Parameters：
- bar_type (BarType) – The bar type for this bar.
- open (Price) – The bars open price.
- high (Price) – The bars high price.
- low (Price) – The bars low price.
- close (Price) – The bars close price.
- volume (Quantity) – The bars volume.
- ts_event (uint64_t) – UNIX timestamp (nanoseconds) when the data event occurred.
- ts_init (uint64_t) – UNIX timestamp (nanoseconds) when the data object was initialized.
- is_revision (bool , default False) – If this bar is a revision of a previous bar with the same ts_event.

**Since Nautilus Trader doesn't provide a Binance Bar Data Lodaer，we need to create Bar list manually.**

In [108]:
df = pd.read_csv(
    filepath_or_buffer=parent_dir / "data/BTCUSDT-1d-2025-01.csv",
    names=[
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_volume",
        "count",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "unknow",
    ],
    dtype={
        "open_time": "int64",
        "open": "float64",
        "high": "float64",
        "low": "float64",
        "close": "float64",
        "volume": "float64",
        "close_time": "int64",
        "quote_volume": "float64",
        "count": "int64",
        "taker_buy_base_volume": "float64",
        "taker_buy_quote_volume": "float64",
        "unknow": "int64",
    }
)
df['timestamp'] = pd.to_datetime(df['close_time'],unit='us',utc=True)
df = df.set_index("timestamp")
df = df.drop(columns=['open_time', 'close_time', 'quote_volume', 'count', 'taker_buy_base_volume', 'taker_buy_quote_volume', 'unknow'])

In [ ]:
from nautilus_trader.test_kit.providers import TestInstrumentProvider
from nautilus_trader.model import InstrumentId, Symbol
from nautilus_trader.model.data import BarType
from nautilus_trader.persistence.wranglers import BarDataWrangler
from nautilus_trader.adapters.binance import (
    BINANCE_VENUE,
)

instrument_id = InstrumentId(symbol=Symbol("BTCUSDT"), venue=BINANCE_VENUE)
bar_type = BarType.from_str(f"{instrument_id.value}-1-DAY-LAST-EXTERNAL")
instrument = TestInstrumentProvider.btcusdt_binance()
wrangler = BarDataWrangler(bar_type, instrument)
bars = wrangler.process(df)

### Write Data

In [110]:
catalog.write_data(bars)

## Query Data

In [115]:
from nautilus_trader.model import Bar
catalog.query(
    data_cls=Bar,
    identifiers=["BTCUSDT.BINANCE"],
    start="2025-01-03",
    end="2025-01-10",
)

[Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,96984.79,98976.91,96100.01,98174.18,15253.829360,1735948799999999000),
 Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,98174.17,98778.43,97514.79,98220.50,8990.056510,1736035199999999000),
 Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,98220.51,98836.85,97276.79,98363.61,8095.637230,1736121599999999000),
 Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,98363.61,102480.00,97920.00,102235.60,25263.433750,1736207999999999000),
 Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,102235.60,102724.38,96181.81,96954.61,32059.875370,1736294399999999000),
 Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,96954.60,97268.65,92500.90,95060.61,33704.678940,1736380799999999000),
 Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,95060.61,95382.32,91203.67,92552.49,34544.836850,1736467199999999000)]